# Fashion Retrieval — CLIP Vector Indexing

Builds searchable FAISS vector indexes for **Config A** and **Config B** of the ablation study.

---

### Embedding formula
```
vec = beta * CLIP_vision(crop) + (1 - beta) * CLIP_language(caption)
```
Vectors are L2-normalised before storage. Inner product search then equals cosine similarity.

---

### Configs built here
| Config | beta | Captions? | CLIP state |
|--------|------|-----------|------------|
| A | 1.0 | No | Frozen |
| B | 0.7 | Yes | Frozen |
| B | 0.5 | Yes | Frozen |

---

### Inputs
- `vr-yolo-bbox-cropped-images` — bbox crops + master_crops.csv
- `blip-captions-data` — gallery_captions.json

### Outputs
- `gallery_vectors_A_b10.npy`, `gallery_vectors_B_b07.npy`, `gallery_vectors_B_b05.npy`
- `idx_A_b10.bin`, `idx_B_b07.bin`, `idx_B_b05.bin`  *(HNSW indexes)*
- `item_index_map.csv`

## 1. Install Packages

In [1]:
!pip uninstall -y faiss faiss-gpu
!pip install ftfy regex transformers faiss-cpu Pillow --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 71.2 MB/s eta 0:00:00


## 2. Imports

In [2]:
import os
import json
import numpy as np
import pandas as pd
import torch
import faiss
from PIL import Image
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel
import warnings
warnings.filterwarnings('ignore')

GPU = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Runtime device  : {GPU}')
if GPU == 'cuda':
    print(f'GPU name        : {torch.cuda.get_device_name(0)}')
print(f'PyTorch version : {torch.__version__}')
print(f'FAISS version   : {faiss.__version__}')
print('Setup complete!')

Runtime device  : cuda
GPU name        : Tesla T4
PyTorch version : 2.10.0+cu128
FAISS version   : 1.13.2
Setup complete!


## 3. Paths and Config

In [3]:
# ── Input datasets ───────────────────────────────────────────────────────────
BBOX_CROPS_DIR   = '/kaggle/input/datasets/akibatra25/vr-yolo-bbox-cropped-images'
CAPTIONS_DIR     = '/kaggle/input/datasets/akibatra25/blip-captions-data'

# ── Output directory ─────────────────────────────────────────────────────────
OUT_DIR = '/kaggle/working'
os.makedirs(OUT_DIR, exist_ok=True)

# ── Model ─────────────────────────────────────────────────────────────────────
CLIP_CKPT = 'openai/clip-vit-base-patch32'

# ── Beta (image-text blend weight) values to index ───────────────────────────
# beta=1.0  → Config A: pure vision embedding
# beta=0.7  → Config B: 70% vision, 30% language
# beta=0.5  → Config B: equal blend
BETA_LIST = [1.0, 0.7, 0.5]

ENCODE_BATCH = 64   # images per CLIP forward pass

for tag, p in [('BBOX_CROPS_DIR', BBOX_CROPS_DIR), ('CAPTIONS_DIR', CAPTIONS_DIR)]:
    ok = 'Found ✓' if os.path.exists(p) else 'NOT FOUND ✗'
    print(f'[{ok}] {tag}: {p}')

[Found ✓] BBOX_CROPS_DIR: /kaggle/input/datasets/akibatra25/vr-yolo-bbox-cropped-images
[Found ✓] CAPTIONS_DIR: /kaggle/input/datasets/akibatra25/blip-captions-data


## 4. Load Gallery CSV and Fix Paths

In [4]:
full_table = pd.read_csv(os.path.join(BBOX_CROPS_DIR, 'master_crops.csv'))
gal_table  = full_table[full_table['split'] == 'gallery'].reset_index(drop=True)

print(f'Total master rows : {len(full_table):,}')
print(f'Gallery rows      : {len(gal_table):,}')

def translate_path(saved_path):
    if pd.isna(saved_path):
        return saved_path
    for pfx in ['/kaggle/working/', '/kaggle/input/']:
        if saved_path.startswith(pfx):
            tail = saved_path.replace(pfx, '')
            for ds_name in ['vr-yolo-bbox-cropped-images/', 'datasets/akibatra25/vr-yolo-bbox-cropped-images/']:
                tail = tail.replace(ds_name, '')
            return os.path.join(BBOX_CROPS_DIR, tail)
    return saved_path

gal_table['img_path'] = gal_table['crop_path'].apply(translate_path)
gal_table['on_disk']  = gal_table['img_path'].apply(
    lambda p: os.path.exists(p) if isinstance(p, str) else False
)

print(f'Gallery crops on disk: {gal_table["on_disk"].sum():,} / {len(gal_table):,}')

if gal_table['on_disk'].sum() < len(gal_table) * 0.9:
    print('Trying direct path construction...')
    def direct_path(img_name):
        rel = img_name[4:] if img_name.startswith('img/') else img_name
        for sub in ['data/bbox_crops', 'data/yolo_crops']:
            p = os.path.join(BBOX_CROPS_DIR, sub, rel)
            if os.path.exists(p): return p
        return os.path.join(BBOX_CROPS_DIR, 'data/bbox_crops', rel)
    gal_table['img_path'] = gal_table['image_name'].apply(direct_path)
    gal_table['on_disk']  = gal_table['img_path'].apply(os.path.exists)
    print(f'After direct path: {gal_table["on_disk"].sum():,} / {len(gal_table):,}')

print('\nSample paths:')
for p in gal_table['img_path'].head(2):
    print(f'  [{"OK" if os.path.exists(p) else "MISSING"}] {p}')

Total master rows : 52,712
Gallery rows      : 12,612
Gallery crops on disk: 12,612 / 12,612

Sample paths:
  [OK] /kaggle/input/datasets/akibatra25/vr-yolo-bbox-cropped-images/data/bbox_crops/WOMEN/Blouses_Shirts/id_00000001/02_1_front.jpg
  [OK] /kaggle/input/datasets/akibatra25/vr-yolo-bbox-cropped-images/data/bbox_crops/WOMEN/Blouses_Shirts/id_00000001/02_3_back.jpg


In [5]:
with open(os.path.join(CAPTIONS_DIR, 'gallery_captions.json')) as fh:
    gal_captions = json.load(fh)

print(f'Gallery captions loaded : {len(gal_captions):,}')
gal_names = set(gal_table['image_name'].tolist())
covered   = gal_names.intersection(set(gal_captions.keys()))
print(f'Captions covering gallery: {len(covered):,} / {len(gal_names):,}')
print('\nSample captions:')
for nm, cap in list(gal_captions.items())[:3]:
    print(f'  {nm.split("/")[-1]}: {cap}')

Gallery captions loaded : 12,612
Captions covering gallery: 12,612 / 12,612

Sample captions:
  02_1_front.jpg: a woman in black shorts and a white blouse
  02_3_back.jpg: the back view of a woman wearing a white blouse
  01_1_front.jpg: a woman wearing a tank top with the words snoop dogg on it


## 5. Load CLIP (Frozen)

In [6]:
print(f'Loading CLIP checkpoint: {CLIP_CKPT}')
clip_proc  = CLIPProcessor.from_pretrained(CLIP_CKPT)
clip_net   = CLIPModel.from_pretrained(CLIP_CKPT).to(GPU)

for p in clip_net.parameters():
    p.requires_grad = False
clip_net.eval()

VEC_DIM = clip_net.config.projection_dim
print(f'CLIP loaded and frozen!')
print(f'Vector dimension: {VEC_DIM}')

Loading CLIP checkpoint: openai/clip-vit-base-patch32


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIP loaded and frozen!
Vector dimension: 512


## 6. Encoding Functions

In [7]:
def batch_encode_images(path_list):
    imgs, ok_idx = [], []
    for i, p in enumerate(path_list):
        try:
            imgs.append(Image.open(p).convert('RGB'))
            ok_idx.append(i)
        except Exception:
            pass
    if not imgs:
        return None, ok_idx
    inp = clip_proc(images=imgs, return_tensors='pt', padding=True).to(GPU)
    with torch.no_grad():
        raw = clip_net.get_image_features(**inp)
        vecs = raw.pooler_output if hasattr(raw, 'pooler_output') and not isinstance(raw, torch.Tensor) else raw
    vecs = vecs / vecs.norm(dim=-1, keepdim=True)
    return vecs.cpu().numpy(), ok_idx


def batch_encode_texts(text_list):
    inp = clip_proc(text=text_list, return_tensors='pt', padding=True, truncation=True, max_length=77).to(GPU)
    with torch.no_grad():
        raw = clip_net.get_text_features(**inp)
        vecs = raw.pooler_output if hasattr(raw, 'pooler_output') and not isinstance(raw, torch.Tensor) else raw
    vecs = vecs / vecs.norm(dim=-1, keepdim=True)
    return vecs.cpu().numpy()


def blend_vectors(img_vecs, txt_vecs, beta):
    blended = beta * img_vecs + (1.0 - beta) * txt_vecs
    norms   = np.linalg.norm(blended, axis=-1, keepdims=True)
    return blended / np.maximum(norms, 1e-8)


print('Encoding functions ready ✓')
# Quick test
_row = gal_table[gal_table['on_disk']].iloc[0]
_ie, _ = batch_encode_images([_row['img_path']])
_te     = batch_encode_texts(['a blue dress'])
print(f'Image vec shape : {_ie.shape}  norm: {np.linalg.norm(_ie[0]):.4f}')
print(f'Text vec shape  : {_te.shape}  norm: {np.linalg.norm(_te[0]):.4f}')

Encoding functions ready ✓
Image vec shape : (1, 512)  norm: 1.0000
Text vec shape  : (1, 512)  norm: 1.0000


## 7. Compute Gallery Image Vectors

In [8]:
valid_gal = gal_table[gal_table['on_disk']].reset_index(drop=True)
N = len(valid_gal)
print(f'Encoding {N:,} gallery images...')

img_matrix = np.zeros((N, VEC_DIM), dtype=np.float32)
bad_idx    = []

for s in tqdm(range(0, N, ENCODE_BATCH), desc='Vision encoding'):
    chunk = valid_gal.iloc[s : s + ENCODE_BATCH]
    vecs, ok = batch_encode_images(chunk['img_path'].tolist())
    if vecs is None:
        bad_idx.extend(range(s, s + len(chunk)))
        continue
    for li, gi in enumerate(ok):
        img_matrix[s + gi] = vecs[li]

print(f'Vision encoding done — shape: {img_matrix.shape}, failed: {len(bad_idx)}')

Encoding 12,612 gallery images...


Vision encoding: 100%|██████████| 198/198 [01:35<00:00,  2.08it/s]

Vision encoding done — shape: (12612, 512), failed: 0


## 8. Compute Gallery Text Vectors

In [9]:
print(f'Encoding {N:,} gallery captions...')
txt_matrix    = np.zeros((N, VEC_DIM), dtype=np.float32)
n_fallback    = 0

for s in tqdm(range(0, N, ENCODE_BATCH), desc='Language encoding'):
    chunk = valid_gal.iloc[s : s + ENCODE_BATCH]
    caps  = []
    for _, r in chunk.iterrows():
        c = gal_captions.get(r['image_name'], '')
        if not c:
            c = 'a clothing item'
            n_fallback += 1
        caps.append(c)
    vecs = batch_encode_texts(caps)
    txt_matrix[s : s + len(caps)] = vecs

print(f'Language encoding done — shape: {txt_matrix.shape}')
print(f'Fallback captions used: {n_fallback}')

Encoding 12,612 gallery captions...


Language encoding: 100%|██████████| 198/198 [00:07<00:00, 25.55it/s]

Language encoding done — shape: (12612, 512)
Fallback captions used: 0


## 9. Build Indexes for Each Beta Value

In [10]:
for beta in BETA_LIST:
    print(f'\n--- Beta = {beta} ---')

    if beta == 1.0:
        cfg_tag  = 'A'
        beta_tag = 'b10'
        fused    = img_matrix.copy()
        print('Config A: vision-only (no language signal)')
    elif beta == 0.7:
        cfg_tag  = 'B'
        beta_tag = 'b07'
        fused    = blend_vectors(img_matrix, txt_matrix, beta)
        print(f'Config B: {beta*100:.0f}% vision + {(1-beta)*100:.0f}% language')
    else:
        cfg_tag  = 'B'
        beta_tag = f'b{int(beta*10):02d}'
        fused    = blend_vectors(img_matrix, txt_matrix, beta)
        print(f'Config B: {beta*100:.0f}% vision + {(1-beta)*100:.0f}% language')

    print(f'Vector matrix shape : {fused.shape}')
    print(f'Sample L2 norm      : {np.linalg.norm(fused[0]):.4f}  (expected ~1.0)')

    # Save raw vectors
    vec_file = os.path.join(OUT_DIR, f'gallery_vectors_{cfg_tag}_{beta_tag}.npy')
    np.save(vec_file, fused)
    print(f'Vectors saved → {vec_file}')

    # Build FAISS HNSW index (Hierarchical Navigable Small World — as per project spec)
    # IndexHNSWFlat uses inner-product space; vectors are L2-normalised so IP == cosine sim
    M         = 32   # number of neighbours per node (controls graph connectivity)
    ef_search = 64   # search-time beam width (higher = more accurate, slower)
    ef_constr = 200  # construction-time beam width (higher = better graph quality)

    # Build HNSW index
    idx_hnsw  = faiss.IndexHNSWFlat(VEC_DIM, M, faiss.METRIC_INNER_PRODUCT)
    idx_hnsw.hnsw.efSearch       = ef_search
    idx_hnsw.hnsw.efConstruction = ef_constr
    idx_hnsw.add(fused.astype(np.float32))
    print(f'HNSW index: {idx_hnsw.ntotal:,} vectors  (M={M}, efSearch={ef_search}, efConstruction={ef_constr})')

    idx_file = os.path.join(OUT_DIR, f'idx_{cfg_tag}_{beta_tag}.bin')
    faiss.write_index(idx_hnsw, idx_file)
    print(f'Index saved → {idx_file}')

print('\nAll indexes built!')



--- Beta = 1.0 ---
Config A: vision-only (no language signal)
Vector matrix shape : (12612, 512)
Sample L2 norm      : 1.0000  (expected ~1.0)
Vectors saved → /kaggle/working/gallery_vectors_A_b10.npy
HNSW index: 12,612 vectors  (M=32, efSearch=64, efConstruction=200)
Index saved → /kaggle/working/idx_A_b10.bin

--- Beta = 0.7 ---
Config B: 70% vision + 30% language
Vector matrix shape : (12612, 512)
Sample L2 norm      : 1.0000  (expected ~1.0)
Vectors saved → /kaggle/working/gallery_vectors_B_b07.npy
HNSW index: 12,612 vectors  (M=32, efSearch=64, efConstruction=200)
Index saved → /kaggle/working/idx_B_b07.bin

--- Beta = 0.5 ---
Config B: 50% vision + 50% language
Vector matrix shape : (12612, 512)
Sample L2 norm      : 1.0000  (expected ~1.0)
Vectors saved → /kaggle/working/gallery_vectors_B_b05.npy
HNSW index: 12,612 vectors  (M=32, efSearch=64, efConstruction=200)
Index saved → /kaggle/working/idx_B_b05.bin

All indexes built!


## 10. Save Item Index Map

In [11]:
index_map = valid_gal[['image_name', 'item_id', 'split', 'clothes_type', 'img_path']].copy()
index_map = index_map.rename(columns={'img_path': 'crop_path'})
index_map['faiss_index_pos'] = range(len(index_map))

map_path = os.path.join(OUT_DIR, 'item_index_map.csv')
index_map.to_csv(map_path, index=False)
print(f'Item index map saved → {map_path}')
print(f'Rows    : {len(index_map):,}')
print(f'Columns : {index_map.columns.tolist()}')

Item index map saved → /kaggle/working/item_index_map.csv
Rows    : 12,612
Columns : ['image_name', 'item_id', 'split', 'clothes_type', 'crop_path', 'faiss_index_pos']


## 11. Sanity Check

In [12]:
import matplotlib.pyplot as plt

chk_index = faiss.read_index(os.path.join(OUT_DIR, 'idx_A_b10.bin'))
chk_vecs  = np.load(os.path.join(OUT_DIR, 'gallery_vectors_A_b10.npy'))

q_row = valid_gal.sample(1, random_state=42).iloc[0]
q_idx = q_row.name
q_vec = chk_vecs[q_idx:q_idx+1].astype(np.float32)

dists, hits = chk_index.search(q_vec, 6)

print(f'Query : {q_row["image_name"].split("/")[-1]}  (item_id={q_row["item_id"]})')
print()
print('Top 6 retrieved:')
for rank, (h, d) in enumerate(zip(hits[0], dists[0])):
    r = index_map.iloc[h]
    tag = '✓ MATCH' if r['item_id'] == q_row['item_id'] else '✗'
    print(f'  Rank {rank+1}: item_id={r["item_id"]}  score={d:.4f}  {tag}')

Query : 04_1_front.jpg  (item_id=id_00004024)

Top 6 retrieved:
  Rank 1: item_id=id_00004024  score=1.0000  ✓ MATCH
  Rank 2: item_id=id_00002056  score=0.9572  ✗
  Rank 3: item_id=id_00002226  score=0.9494  ✗
  Rank 4: item_id=id_00007769  score=0.9492  ✗
  Rank 5: item_id=id_00007846  score=0.9463  ✗
  Rank 6: item_id=id_00004551  score=0.9456  ✗


## 12. Summary

In [13]:
print('=' * 60)
print('    CLIP INDEXING COMPLETE')
print('=' * 60)
print(f'  CLIP checkpoint     : {CLIP_CKPT}')
print(f'  Gallery vectors     : {len(valid_gal):,}')
print(f'  Vector dimension    : {VEC_DIM}')
print()
print('  Output files:')
for fn in sorted(os.listdir(OUT_DIR)):
    if fn.endswith(('.bin', '.npy', '.csv')):
        sz = os.path.getsize(os.path.join(OUT_DIR, fn)) / 1e6
        print(f'    {fn}  ({sz:.1f} MB)')
print()
print('  Next steps:')
print('    1. Save output as Kaggle dataset (e.g. clip-indexes-ab-friend)')
print('    2. Run evaluation notebook')
print('    3. Run CLIP fine-tuning notebook for Config C')
print('=' * 60)

    CLIP INDEXING COMPLETE
  CLIP checkpoint     : openai/clip-vit-base-patch32
  Gallery vectors     : 12,612
  Vector dimension    : 512

  Output files:
    gallery_vectors_A_b10.npy  (25.8 MB)
    gallery_vectors_B_b05.npy  (25.8 MB)
    gallery_vectors_B_b07.npy  (25.8 MB)
    idx_A_b10.bin  (29.3 MB)
    idx_B_b05.bin  (29.3 MB)
    idx_B_b07.bin  (29.3 MB)
    item_index_map.csv  (2.5 MB)

  Next steps:
    1. Save output as Kaggle dataset (e.g. clip-indexes-ab-friend)
    2. Run evaluation notebook
    3. Run CLIP fine-tuning notebook for Config C
